# 1. Supervised stages: SFT, DPO, and the hedge-seeded SFT

* **Pipeline and experiment 1:** Qwen2.5-0.5B (base) is evaluated, fine-tuned on GSM8K solutions (SFT), then trained with DPO on pairs of its own correct and incorrect answers.
* **Experiment 2:** a second SFT run in which 10% of the targets end in a hedge (`#### 24 or 72`).

Every checkpoint is evaluated on GSM8K test and SVAMP.

**Before running:** *Runtime > Change runtime type > T4 GPU*. Expected time: about 2 hours.

If Colab disconnects, reconnect and use *Runtime > Run all* again: finished stages and evaluations are skipped, and GRPO resumes from its last checkpoint (saved every 25 steps).

In [ ]:
# Runs on a free Colab T4. Checkpoints and logs go to Google Drive, so a disconnect loses at most a few minutes.
REPO_URL = "https://github.com/lobby9000/reward-hacking-gsm8k"
DRIVE_DIR = "/content/drive/MyDrive/reward-hacking-gsm8k"

import os
from google.colab import drive

drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/runs", exist_ok=True)
if not os.path.exists("/content/reward-hacking-gsm8k"):
    !git clone -q {REPO_URL} /content/reward-hacking-gsm8k
%cd /content/reward-hacking-gsm8k
!git pull -q
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
if not os.path.islink("runs"):
    !rm -rf runs && ln -s {DRIVE_DIR}/runs runs
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Unit and smoke tests (CPU, about 15 seconds). They check padding, log-probabilities, losses, graders and monitors before any GPU time is spent.

In [ ]:
!python -m pytest -q

## Baseline: the base model with 4 worked examples (about 10 minutes)

In [ ]:
!python -m posttrain.evaluate --config configs/eval.yaml --set name=base --set fewshot=4 --set "conditions=[clean]" --set max_new_tokens=320

## Stage 1: SFT (about 20 minutes)

In [ ]:
!python -m posttrain.sft --config configs/sft.yaml

In [ ]:
!python -m posttrain.evaluate --config configs/eval.yaml --set name=sft --set "model.adapters=[runs/sft/adapter]"

## Stage 2: DPO (about 30 minutes, mostly sampling the preference pairs)

In [ ]:
!python -m posttrain.dpo --config configs/dpo.yaml

In [ ]:
!python -m posttrain.evaluate --config configs/eval.yaml --set name=dpo --set "model.adapters=[runs/sft/adapter,runs/dpo/adapter]"

## Experiment 2: SFT with seeded hedges (about 25 minutes)

In [ ]:
!python -m posttrain.sft --config configs/sft_hedge.yaml

In [ ]:
!python -m posttrain.evaluate --config configs/eval_hedge.yaml --set name=sft_hedge --set "model.adapters=[runs/sft_hedge/adapter]"

Next: `02_grpo.ipynb`, once for each GRPO run.